<table><tr>
<td width="76"><div align="center" style="font-size:44px">🔌</div></td>
<td><h1 style="margin:0">LAB 0 · Encender el laboratorio</h1>
<b>Máster en Cyber Threat Intelligence · Módulo 06 · Sesión 44 — Análisis estático de malware</b><br>
⏱️ <b>5 minutos</b> &nbsp;·&nbsp; 🎯 Que todo el mundo tenga el entorno funcionando antes de empezar</td>
</tr></table>

---

## ¿Qué es esto que estoy mirando?

Esto es un **Google Colab**: un cuaderno con texto y con "celdas" de código que se ejecutan
en **un ordenador de Google, no en el tuyo**.

Eso nos viene perfecto hoy por tres razones:

| | |
|---|---|
| 🧪 | Es una **máquina virtual desechable**: cuando cierras la pestaña, desaparece con todo lo que hubiera dentro. |
| 🛡️ | El malware **nunca toca tu equipo**. Tu antivirus no se va a volver loco. |
| 🐧 | Por debajo es un **Linux** con todas las herramientas de análisis instaladas en 30 segundos. |

**No necesitas saber programar.** Cada celda tiene un botón ▶️ a la izquierda: lo pulsas y ya.
Cuando te toque escribir algo te lo voy a marcar bien claro, y siempre tendrás la solución justo debajo.

---
## 🛑 Antes de empezar: 4 reglas

**1 · Esto es malware real.**
Trabajamos siempre dentro de Colab, que es una máquina virtual de Google
que se destruye cuando cierras la pestaña.

**2 · No descargues nada a tu ordenador.**
Ni las muestras, ni los ZIP, ni los ficheros que generes.

**3 · Nunca lo ejecutes.**
Todo lo que vamos a hacer hoy es *mirar* el fichero sin abrirlo.
Eso es exactamente el **análisis estático**.

**4 · ¿No sabes programar? No pasa nada.**
Todas las celdas se ejecutan con el botón ▶️ de la izquierda.
Donde te toque escribir algo te lo indico, y tienes la solución justo debajo.

---
## Paso 1 · Guarda tu propia copia (recomendado)

Arriba a la izquierda: **Archivo → Guardar una copia en Drive**.

Así puedes escribir en las celdas, romper cosas y volver a ello esta noche sin miedo.
Si no lo haces tampoco pasa nada: el notebook funciona igual, pero no se guardarán tus cambios.

---
## Paso 2 · Arranca el laboratorio

Pulsa ▶️ en la celda de abajo y espera unos **40 segundos**. Instala las herramientas,
descarga las muestras y las descomprime. No hace falta que entiendas lo que hace.

> Google te avisará de que *"este cuaderno no lo ha creado Google"*.
> Es normal: dale a **Ejecutar de todos modos**.

> 📦 La segunda celda, **PLAN B**, solo hace falta si la primera no consigue las muestras.
> Si la primera termina con el listado de ficheros, ignórala y sigue.

---
### ⚠️ Esto hay que repetirlo en cada laboratorio

Los laboratorios son **cuadernos distintos**, y Colab le da a cada uno su propia máquina.
Lo que instales aquí **no viaja** al LAB 1. En cada cuaderno tendrás que volver a ejecutar
su celda de preparación: son otros 40 segundos y ya está.


In [ ]:
#@title ▶️ EJECUTA ESTA CELDA (botón ▶ a la izquierda) y espera ~40 segundos { display-mode: "form" }

#@markdown ---
#@markdown **No hace falta que entiendas este código todavía.** Solo prepara el laboratorio:
#@markdown instala las herramientas, descarga las muestras y las descomprime.
#@markdown ---

SAMPLES_URL = "https://github.com/jstnk9/kschool_ejercicios/raw/main/analisis_estatico/muestras_kschool.zip" #@param {type:"string"}
PASSWORD    = "infected" #@param {type:"string"}

import os, glob, subprocess

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("1/4  Instalando herramientas de análisis...")
_sh("pip install -q pyzipper pefile oletools yara-python py-tlsh")
print("     ✔ pyzipper, pefile, oletools, yara-python, tlsh")

print("2/4  Consiguiendo las muestras...")
DEST = "/content/muestras_kschool.zip"

if not SAMPLES_URL.strip():
    # Sin URL configurada: las muestras se suben a mano con la celda de abajo.
    print("     ℹ  Este cuaderno no trae URL de descarga.")
    print("        Ve a la celda de abajo, 'PLAN B', y sube el fichero")
    print("        muestras_kschool.zip que te ha pasado el profesor.")
    print("        Es un paso normal: tarda 10 segundos.")
    ok_zip = False
else:
    url = SAMPLES_URL.strip()

    # GitHub sirve DOS urls distintas para el mismo fichero:
    #   .../blob/...  -> la pagina web que lo muestra  (HTML)
    #   .../raw/...   -> el fichero de verdad
    # Si te has copiado la de la barra del navegador, la arreglamos aqui.
    if "github.com" in url and "/blob/" in url:
        url = url.replace("/blob/", "/raw/")
        print("     ℹ  URL de GitHub corregida: /blob/ -> /raw/")
    url = url.replace("?raw=true", "").replace("?raw=1", "")

    if os.path.exists(DEST):
        os.remove(DEST)      # por si un intento anterior dejo un fichero malo

    if "drive.google.com" in url:
        _sh("pip install -q gdown")
        _sh("gdown --fuzzy '" + url + "' -O " + DEST)
    else:
        _sh("wget -q --no-check-certificate '" + url + "' -O " + DEST)

    # Comprobamos que lo descargado es DE VERDAD un ZIP mirando sus primeros
    # bytes. Que es, mira tu por donde, justo lo que vas a aprender hoy:
    # un ZIP siempre empieza por 50 4B 03 04, o sea "PK".
    cabecera = open(DEST, "rb").read(4) if os.path.exists(DEST) else b""
    ok_zip = cabecera == b"PK\x03\x04"

    if ok_zip:
        print("     ✔ Descargado (" + str(os.path.getsize(DEST)//1024) + " KB, empieza por 'PK' ✔)")
    else:
        print("     ✖ Lo que he descargado NO es un ZIP.")
        print("        Empieza por los bytes " + (cabecera.hex() or "(nada)") +
              " y un ZIP empieza siempre por 504b0304.")
        if b"<" in cabecera or b"\n" in cabecera:
            print("")
            print("        Parece una pagina HTML. Lo tipico: la URL apunta a la PAGINA")
            print("        de GitHub y no al fichero. Fijate en la diferencia:")
            print("           .../blob/main/...  <- pagina web   ✖")
            print("           .../raw/main/...   <- el fichero   ✔")
            print("        Pulsa el boton 'Raw' en GitHub y copia esa URL.")
        print("")
        print("        Alternativa: usa la celda de abajo, 'PLAN B'.")

print("3/4  Descomprimiendo (contraseña: infected)...")
import pyzipper
if ok_zip:
    try:
        with pyzipper.AESZipFile(DEST) as z:
            z.setpassword(PASSWORD.encode())
            z.extractall("/content/")
        print("     ✔ Descomprimido en /content/muestras/")
    except Exception as e:
        print("     ✖ Error al descomprimir:", e)

print("4/4  Comprobando el laboratorio...")
MUESTRAS = "/content/muestras"
ficheros = sorted(f for f in glob.glob(MUESTRAS + "/*") if not f.endswith("LEEME.txt"))
if len(ficheros) >= 9:
    print("     ✔ " + str(len(ficheros)) + " muestras listas")
    print("")
    print("=" * 52)
    print("   LABORATORIO LISTO  ✅")
    print("=" * 52)
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("     ✖ Solo encuentro " + str(len(ficheros)) + " ficheros. Usa la celda 'PLAN B' de abajo.")

print("""
⚠️  RECUERDA: esto son muestras REALES de malware.
    Estás dentro de una máquina virtual de Google que se destruye al cerrar.
    NO descargues estos ficheros a tu ordenador. NO los ejecutes.
    Hoy solo vamos a MIRARLOS, que es justo de lo que va el análisis estático.
""")

In [ ]:
#@title 📦 PLAN B — solo si la celda de arriba no ha conseguido las muestras { display-mode: "form" }
import glob, os

# Esta celda se puede ejecutar sola, sin haber pasado por la de arriba,
# asi que se instala ella misma lo que necesita.
try:
    import pyzipper
except ImportError:
    print("Instalando pyzipper (5 segundos)...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyzipper"], check=False)
    import pyzipper

ZIP_YA_SUBIDO = "/content/muestras_kschool.zip"

if os.path.exists(ZIP_YA_SUBIDO):
    # ya lo habias subido con el panel de Archivos de la izquierda
    print("Encontrado", ZIP_YA_SUBIDO, "- no hace falta que lo subas otra vez.")
    nombres = [ZIP_YA_SUBIDO]
else:
    from google.colab import files
    print("Pulsa en 'Elegir archivos' y selecciona muestras_kschool.zip")
    nombres = list(files.upload())

for nombre in nombres:
    try:
        with pyzipper.AESZipFile(nombre) as z:
            z.setpassword(b"infected")
            z.extractall("/content/")
    except Exception as e:
        print("✖ No he podido abrir", nombre, "->", e)

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))
if ficheros:
    print("")
    print("✔ " + str(len(ficheros)) + " muestras listas en /content/muestras/")
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("✖ Sigo sin ver las muestras. Avisa en el chat.")

---
## Paso 3 · Comprobación (30 segundos)

Si arriba te ha salido **LABORATORIO LISTO ✅** y una lista de 9 ficheros, ya estás.

Ejecuta esta última celda y **escribe el resultado en el chat de la clase**.
Así el profesor sabe quién está listo y quién necesita ayuda.

In [ ]:
import glob, os

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))
total    = sum(os.path.getsize(f) for f in ficheros)

print(f"Tengo {len(ficheros)} muestras y ocupan {total/1024/1024:.2f} MB")
print("El fichero más grande es:", os.path.basename(max(ficheros, key=os.path.getsize)))
print()
print("👉 Copia estas dos líneas en el chat de la clase")

> **Respuesta correcta:** 9 muestras · 3,19 MB · el más grande es `actualizacion_flash_DESEMPAQUETADO.bin`.
>
> Si te sale eso, **estás listo**. Si te sale otra cosa o un error rojo, dilo en el chat ahora,
> que es mucho mejor arreglarlo ahora que dentro de veinte minutos.

---
## Mientras esperamos al resto…

Echa un ojo a los nombres de los ficheros que te ha listado la celda de setup.

Son los nombres tal y como llegarían en un correo. Piensa un segundo (no hace falta que hagas nada):

- ¿Cuál de ellos abrirías sin pensártelo dos veces?
- ¿Alguno te da mala espina solo por el nombre?

Guárdate la respuesta. En el **LAB 1** vamos a descubrir que **casi ninguno es lo que dice ser**.